# Data Analytics Lab Sheet 7: Customer Segmentation using K-Means Clustering

**Name:** ANURAG PARMAR  
**Roll No.:** 8  
**Section:** B  
**CU ID:** CU24250076  
**Program:** B.Tech CSE  

## Aim
Create a reproducible customer dataset, scale its features, apply K-Means, and visualize customer segments.

> Run cells from top to bottom. Dataset sources and any synthetic-data notes are documented in the relevant sections.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
sns.set_theme(style="whitegrid")

## 2. Create customer dataset

In [ ]:
rng = np.random.default_rng(42)
n = 200
df = pd.DataFrame({
    "CustomerID": np.arange(1, n + 1),
    "Age": rng.integers(18, 71, n),
    "Annual Income (k$)": rng.integers(15, 131, n),
    "Spending Score (1-100)": rng.integers(1, 101, n)
})
display(df.head())
print("Dataset shape:", df.shape)

## 3. Inspect and prepare

In [ ]:
print(df.dtypes)
display(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
features = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
X = df[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 4. Choose K using inertia and silhouette score

In [ ]:
inertias, silhouettes = [], []
k_values = range(2, 8)
for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    inertias.append(model.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(list(k_values), inertias, marker="o")
ax[0].set_title("Elbow Method")
ax[0].set_xlabel("Number of clusters")
ax[0].set_ylabel("Inertia")
ax[1].plot(list(k_values), silhouettes, marker="o")
ax[1].set_title("Silhouette Score")
ax[1].set_xlabel("Number of clusters")
plt.tight_layout()
plt.show()

## 5. Fit K-Means and visualize segments

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X_scaled)
print("Silhouette score:", round(silhouette_score(X_scaled, df["Cluster"]), 4))
display(df.groupby("Cluster")[features].mean().round(2))
sns.scatterplot(data=df, x="Annual Income (k$)", y="Spending Score (1-100)", hue="Cluster", palette="tab10")
plt.title("Customer Segments")
plt.tight_layout()
plt.show()

## 6. Export segmented dataset

In [ ]:
df.to_csv("customer_segments.csv", index=False)
print("Saved customer_segments.csv")